# MiniMax H3 · 两套启动入口 → TapNow Local

两套权重都已完整备份到 **mert 的 Google Drive**。每次只选一套启动，无需重新下载。

### A · 全能参考（Ref2VA）
图片、视频、音频可以混合参考。画布选择 **Colab → 全能参考**。

### B · 首尾帧／文字（FL2VA）
支持首帧、尾帧或首尾两张图片，也支持不带参考图的文字生成。画布选择 **Colab → 首尾帧**，或 **文字**。

## 每次使用的步骤
1. 右上角选择 **mert**；「代码执行程序 → 更改运行时类型」使用 **2026.07、G4 GPU、高 RAM**。
2. 在下面 **① 两套启动入口** 的下拉框选择 A 或 B，并勾选账号确认项。切换模型前先结束旧会话。
3. 按顺序运行 **② 挂载 Drive → 部署脚本 → ③ 恢复所选模型 → ④ 启动服务 → ⑤ 获取本次 HTTPS 地址**。只读进度检查是可选项。
4. 把 **⑤ 的新输出地址**复制到 TapNow「模型连接 → Colab」，沿用相同 Token，点击「保存并连接」。旧输出地址不可直接复用。
5. 所有任务结束、结果保存到本地并确认后，默认闲置 **60 秒**自动释放 GPU；TapNow 已保存的自动释放偏好也会在连接或生成时应用。

**Drive 缓存：** `MyDrive/TapNowLocal/models/MiniMax-H3/42ed227ee7df40d41602854ae760620d6eb651fe/`。清单包含 `FL2VA` 和 `Ref2VA` 两个完整分区。相同的文本编码器及 VAE 权重通过清单共用，不要单独删除 FL2VA 文件夹。

恢复和加载也会占用 GPU 时间。当前原始权重配置使用 80 GB 级 GPU、至少 120 GiB 系统内存；不包含独立生图或完整托管 2K 工作流。Ref2VA 的完整出片验证已按用户要求跳过。


In [ ]:
# @title ① 两套启动入口：选择本次要用的模型
# @markdown **选择本次要启动的一套模型：**
MODEL_SET = "全能参考（Ref2VA）" # @param ["全能参考（Ref2VA）", "首尾帧／文字（FL2VA）"]
# @markdown **确认右上角及 Drive 授权账号都是 mert，再勾选下项：**
CONFIRMED_MERT_ACCOUNT = False # @param {type:"boolean"}
# @markdown **生成结果全部保存确认后，自动关闭 Colab 并释放 GPU：**
AUTO_RELEASE = True # @param {type:"boolean"}
# @markdown **保存确认后等待的秒数（默认 60）：**
IDLE_SECONDS = 60 # @param {type:"integer"}

# CONFIRMED_MERT_ACCOUNT：核对右上角和挂载授权账号为 mert 后勾选。
# AUTO_RELEASE：全部任务结束且本地保存确认后，闲置指定秒数释放 GPU。
# 两套均已缓存到 Drive；日常不要开启重新下载或安装。
MODE = {"全能参考（Ref2VA）": "ref2va", "首尾帧／文字（FL2VA）": "fl2va"}[MODEL_SET]
ALLOW_DOWNLOAD = False
INSTALL_ENV = False
SAVE_ENV = False
OPEN_AUTHENTICATED_TUNNEL = True
assert CONFIRMED_MERT_ACCOUNT, '请确认右上角与挂载授权账号都是 mert，再勾选账号确认项'
assert IDLE_SECONDS >= 30, '自动释放闲置时间至少 30 秒'
print('本次启动：', MODEL_SET)
print('权重来源：mert Drive 已有缓存；仅恢复本次所选的一套')


In [ ]:
# @title ② 挂载 mert Drive（两套模型共用）
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
print('部署目录存在：', Path('/content/drive/MyDrive/TapNowLocal').exists())
print('环境备份存在：', Path('/content/drive/MyDrive/TapNowLocal/session/h3-venv.tar.gz').exists())


In [ ]:
# 只读进度检查：不会启动下载、推理或释放 GPU
from pathlib import Path
import re, shutil
print('本地剩余 GiB:', round(shutil.disk_usage('/content').free/1024**3,1))
model_log=Path('/content/h3-model.log')
if model_log.exists():
    log_text=model_log.read_text(errors='replace')
    progress=re.findall(r'[0-9]+%\|[^\r\n]{0,130}',log_text)
    if progress:
        print('最新加载/生成进度:',progress[-1].split('[2026-')[0])
    events=[line for line in log_text.splitlines() if 'Stage]' in line or 'Application startup complete' in line or 'Error' in line or 'Exception' in line]
    print(chr(10).join(events[-6:]))
else:
    for name in ('ref2va-cache','h3-prepare'):
        p=Path('/content/'+name+'.log')
        if p.exists():
            print(name+':',p.read_text(errors='replace')[-1200:])


内嵌脚本由项目文件统一生成，避免笔记本和独立脚本版本不一致。

In [ ]:
import pathlib, sys
h3_files = {'auth_proxy.py': '"""Expose only authenticated generation routes; keep SGLang management private."""\nimport hmac\nimport os\nimport re\n\n\ndef allowed_route(method, path):\n    return (method == \'GET\' and path in (\'/health\', \'/v1/models\')) or (\n        method == \'POST\' and path == \'/v1/videos\'\n    ) or (method == \'GET\' and re.fullmatch(r\'/v1/videos/[A-Za-z0-9_-]{1,100}(?:/content)?\', path) is not None)\n\n\ndef authorized(header, token):\n    return bool(token) and hmac.compare_digest(header.encode(), (\'Bearer \' + token).encode())\n\n\ndef create_app():\n    import httpx\n    from contextlib import asynccontextmanager\n    from fastapi import FastAPI, Request\n    from fastapi.responses import JSONResponse, StreamingResponse\n\n    token = os.environ.get(\'H3_PROXY_TOKEN\', \'\')\n    if len(token) < 32:\n        raise RuntimeError(\'Set H3_PROXY_TOKEN to a random token of at least 32 characters\')\n\n    @asynccontextmanager\n    async def lifespan(app):\n        async with httpx.AsyncClient(base_url=\'http://127.0.0.1:30020\', timeout=60, trust_env=False) as client:\n            app.state.upstream = client\n            yield\n\n    app = FastAPI(lifespan=lifespan, docs_url=None, redoc_url=None, openapi_url=None)\n\n    @app.api_route(\'/{route:path}\', methods=[\'GET\', \'POST\', \'PUT\', \'PATCH\', \'DELETE\', \'OPTIONS\'])\n    async def proxy(route: str, request: Request):\n        if not authorized(request.headers.get(\'authorization\', \'\'), token):\n            return JSONResponse({\'error\': \'Unauthorized\'}, status_code=401)\n        path = \'/\' + route\n        if not allowed_route(request.method, path):\n            return JSONResponse({\'error\': \'Route not exposed\'}, status_code=404)\n        body = bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body) > 1024 * 1024:\n                return JSONResponse({\'error\': \'Request too large\'}, status_code=413)\n        try:\n            upstream_request = request.app.state.upstream.build_request(\n                request.method, path, content=bytes(body),\n                headers={\'Content-Type\': request.headers.get(\'content-type\', \'application/json\'),\n                         \'Accept-Encoding\': \'identity\'},\n            )\n            upstream = await request.app.state.upstream.send(upstream_request, stream=True)\n        except httpx.HTTPError:\n            return JSONResponse({\'error\': \'Model service unavailable\'}, status_code=503)\n        headers = {key: value for key, value in upstream.headers.items()\n                   if key in (\'content-type\', \'content-length\', \'content-disposition\', \'content-encoding\')}\n        async def stream():\n            try:\n                async for chunk in upstream.aiter_raw():\n                    yield chunk\n            finally:\n                await upstream.aclose()\n        return StreamingResponse(stream(), status_code=upstream.status_code, headers=headers)\n\n    return app\n\n\nif __name__ == \'__main__\':\n    import sys\n    if \'--self-test\' in sys.argv:\n        assert allowed_route(\'GET\', \'/v1/videos/video_123/content\')\n        assert allowed_route(\'POST\', \'/v1/videos\')\n        for method, path in [(\'DELETE\', \'/v1/videos/x\'), (\'GET\', \'/server_info\'),\n                             (\'GET\', \'/v1/videos/../../server_info\'), (\'POST\', \'/update_weights\')]:\n            assert not allowed_route(method, path)\n        assert authorized(\'Bearer \' + \'x\' * 32, \'x\' * 32)\n        assert not authorized(\'Bearer wrong\', \'x\' * 32)\n        assert not authorized(\'Bearer \', \'\')\n        print(\'proxy policy checks passed\')\n    else:\n        import uvicorn\n        uvicorn.run(create_app(), host=\'127.0.0.1\', port=30011, access_log=False)\n', 'benchmark.py': '"""Measure native H3 submit-to-downloaded-MP4 latency without the canvas UI."""\nimport argparse\nimport json\nimport os\nimport pathlib\nimport re\nimport time\nimport urllib.error\nimport urllib.request\n\nPROMPT = \'At dawn in a pine forest, a red fox walks slowly through thin mist. A low camera tracks forward smoothly. Soft backlight, realistic fur. Gentle wind and birdsong.\'\nPAYLOAD = {\n    \'model\': \'MiniMaxAI/MiniMax-H3\', \'prompt\': PROMPT, \'seconds\': 4,\n    \'task\': \'t2va\', \'conditions\': [],\n    \'target\': {\'short_edge\': 768, \'aspect_ratio\': \'16:9\', \'duration_seconds\': 4.0},\n    \'num_outputs_per_prompt\': 1, \'num_inference_steps\': 50,\n    \'flow_shift\': 12.0, \'audio_flow_shift\': 3.0, \'seed\': 42, \'quality\': \'lossless\',\n}\n\n\ndef request(base, route, token, payload=None):\n    headers = {\'Authorization\': \'Bearer \' + token} if token else {}\n    body = None\n    if payload is not None:\n        headers[\'Content-Type\'] = \'application/json\'\n        body = json.dumps(payload).encode()\n    return urllib.request.urlopen(urllib.request.Request(base + route, data=body, headers=headers), timeout=60)\n\n\ndef measure(base, output, label, token):\n    started = time.perf_counter()\n    with request(base, \'/v1/videos\', token, PAYLOAD) as response:\n        job = json.load(response)\n    submitted = time.perf_counter()\n    job_id = job[\'id\']\n    if not isinstance(job_id, str) or not re.fullmatch(r\'[A-Za-z0-9_-]{1,100}\', job_id):\n        raise ValueError(\'Invalid remote job ID\')\n    while job[\'status\'] != \'completed\':\n        if job[\'status\'] == \'failed\':\n            raise RuntimeError(json.dumps(job.get(\'error\', job), ensure_ascii=False))\n        if time.perf_counter() - started > 1800:\n            raise TimeoutError(\'30 minutes elapsed; remote GPU work may still be running\')\n        time.sleep(.5)\n        with request(base, \'/v1/videos/\' + job_id, token) as response:\n            job = json.load(response)\n    generated = time.perf_counter()\n    target = output / (label + \'.mp4\')\n    temp = target.with_suffix(\'.mp4.part\')\n    try:\n        with request(base, \'/v1/videos/\' + job_id + \'/content\', token) as response, temp.open(\'wb\') as dest:\n            size = 0\n            while chunk := response.read(1024 * 1024):\n                size += len(chunk)\n                if size > 250 * 1024 * 1024:\n                    raise ValueError(\'Video exceeds 250 MiB\')\n                dest.write(chunk)\n        with temp.open(\'rb\') as video:\n            header = video.read(12)\n        if size < 12 or header[4:8] != b\'ftyp\':\n            raise ValueError(\'Response is not a valid MP4 container\')\n        temp.replace(target)\n    except BaseException:\n        temp.unlink(missing_ok=True)\n        raise\n    finished = time.perf_counter()\n    result = {\'label\': label, \'request\': PAYLOAD, \'job\': job,\n              \'submitMs\': round((submitted-started)*1000),\n              \'generationAndPollingMs\': round((generated-submitted)*1000),\n              \'downloadMs\': round((finished-generated)*1000),\n              \'totalMs\': round((finished-started)*1000), \'bytes\': size,\n              \'scope\': \'API submit to complete MP4 download; excludes browser decode\',\n              \'pollIntervalMs\': 500, \'measuredAt\': time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime())}\n    (output / (label + \'.json\')).write_text(json.dumps(result, indent=2), encoding=\'utf8\')\n    print(json.dumps({k: result[k] for k in (\'label\', \'submitMs\', \'generationAndPollingMs\', \'downloadMs\', \'totalMs\', \'bytes\')}))\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--base\', default=\'http://127.0.0.1:30020\')\n    parser.add_argument(\'--output\', default=\'/content/h3-benchmarks\')\n    parser.add_argument(\'--label\', default=\'cold-standard-4s-768p\')\n    parser.add_argument(\'--self-test\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.self_test:\n        import io\n        import tempfile\n        from unittest.mock import patch\n        with tempfile.TemporaryDirectory() as directory:\n            output = pathlib.Path(directory)\n            with patch(__name__ + \'.request\', side_effect=[\n                io.BytesIO(b\'{"id":"video_1","status":"completed"}\'), io.BytesIO(b\'<html>error</html>\')\n            ]):\n                try:\n                    measure(\'http://mock\', output, \'invalid\', \'\')\n                    raise AssertionError(\'HTML must not produce a completed benchmark\')\n                except ValueError:\n                    assert not list(output.iterdir())\n            with patch(__name__ + \'.request\', return_value=io.BytesIO(b\'{"id":"../../secret","status":"completed"}\')):\n                try:\n                    measure(\'http://mock\', output, \'invalid-id\', \'\')\n                    raise AssertionError(\'Unsafe job ID must be rejected\')\n                except ValueError:\n                    assert not list(output.iterdir())\n        print(\'benchmark failure-path checks passed (no GPU inference)\')\n    else:\n        output = pathlib.Path(args.output)\n        output.mkdir(parents=True, exist_ok=True)\n        if not re.fullmatch(r\'[A-Za-z0-9_-]{1,100}\', args.label):\n            raise SystemExit(\'Label must use only letters, digits, hyphens, or underscores\')\n        measure(args.base.rstrip(\'/\'), output, args.label, os.environ.get(\'H3_PROXY_TOKEN\', \'\'))\n', 'cache_model.py': '"""Pin a native H3 partition, use local SSD, and reuse verified shared Drive weights."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\n\nREPO = \'MiniMaxAI/MiniMax-H3\'\nREVISION = \'42ed227ee7df40d41602854ae760620d6eb651fe\'\n\n\ndef cache_source(root, name, entry):\n    relative = Path(entry.get(\'source_path\', name))\n    if relative.is_absolute() or \'..\' in relative.parts or not relative.parts:\n        raise RuntimeError(\'Invalid Drive cache source path\')\n    return root / relative\n\n\ndef copy_verified(source, target, expected_size, expected_sha=None):\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temp = target.with_name(target.name + \'.part\')\n    digest = hashlib.sha256()\n    size = 0\n    try:\n        with source.open(\'rb\') as src, temp.open(\'wb\') as dst:\n            while chunk := src.read(8 * 1024 * 1024):\n                size += len(chunk)\n                digest.update(chunk)\n                dst.write(chunk)\n        actual_sha = digest.hexdigest()\n        if size != expected_size or (expected_sha and actual_sha != expected_sha):\n            raise ValueError(\'Model cache checksum mismatch: \' + str(source))\n        temp.replace(target)\n        return actual_sha\n    except BaseException:\n        temp.unlink(missing_ok=True)\n        raise\n\n\ndef wait_for_local_space(need_bytes, path=\'/content\', headroom=15 * 1024**3, timeout=3600):\n    """Colab\'s Drive FUSE stages writes on the local disk; wait until uploads drain."""\n    deadline = time.time() + timeout\n    while shutil.disk_usage(path).free < need_bytes + headroom:\n        if time.time() > deadline:\n            raise RuntimeError(\'Local disk did not drain within timeout; Drive upload may be stuck\')\n        print(f\'Waiting for Drive upload to drain: free {shutil.disk_usage(path).free / 1024**3:.1f} GiB, \'\n              f\'need {(need_bytes + headroom) / 1024**3:.1f} GiB\', flush=True)\n        time.sleep(30)\n\n\ndef cache(local, drive, partition="FL2VA"):\n    from huggingface_hub import HfApi, snapshot_download, hf_hub_download\n    if not Path(\'/content/drive/MyDrive\').is_dir():\n        raise RuntimeError(\'Mount the intended Google Drive before caching weights\')\n    local.mkdir(parents=True, exist_ok=True)\n    drive.mkdir(parents=True, exist_ok=True)\n    api = HfApi()\n    files = [f for f in api.list_repo_tree(REPO, path_in_repo=partition, recursive=True, revision=REVISION)\n             if hasattr(f, \'size\') and f.size > 0]\n    metadata = {f.path: {\'size\': f.size, \'sha256\': f.lfs.sha256 if f.lfs else None} for f in files}\n    total = sum(f[\'size\'] for f in metadata.values())\n    missing = sum(f[\'size\'] for name, f in metadata.items()\n                  if not (local/name).is_file() or (local/name).stat().st_size != f[\'size\'])\n    if shutil.disk_usage(local).free < missing + 8 * 1024**3:\n        raise RuntimeError(f\'Need {missing / 1024**3:.1f} GiB for weights plus 8 GiB free headroom\')\n    print(f\'Pinned {REPO}@{REVISION}; {partition} only: {total / 1024**3:.2f} GiB\', flush=True)\n    print(\'Drive free space is not inferred from FUSE statvfs; a quota error fails this backup.\', flush=True)\n    drive_manifest = drive/\'verified-manifest.json\'\n    prior = json.loads(drive_manifest.read_text()) if drive_manifest.is_file() else {}\n    verified = prior.get(\'files\', {}) if prior.get(\'revision\') == REVISION else {}\n    completed = prior.get(\'completed_partitions\', [prior.get(\'partition\', \'FL2VA\')] if prior.get(\'complete\') else []) if prior.get(\'revision\') == REVISION else []\n    # Reuse only exact official SHA-256 matches from the other verified partition.\n    # source_path points to one canonical Drive file; runtime paths stay partition-specific.\n    by_digest = {}\n    for name, entry in verified.items():\n        if entry.get(\'sha256\'):\n            by_digest.setdefault((entry.get(\'size\'), entry[\'sha256\']), (name, entry))\n    shared_bytes = 0\n    for name, meta in metadata.items():\n        if name in verified or not meta[\'sha256\']:\n            continue\n        shared = by_digest.get((meta[\'size\'], meta[\'sha256\']))\n        if shared:\n            shared_name, entry = shared\n            source = cache_source(drive, shared_name, entry)\n            if source.is_file() and source.stat().st_size == meta[\'size\']:\n                verified[name] = {**meta, \'source_path\': str(source.relative_to(drive))}\n                shared_bytes += meta[\'size\']\n    if shared_bytes:\n        print(f\'Reusing {shared_bytes / 1024**3:.2f} GiB of identical verified Drive weights\', flush=True)\n    restore = [name for name, meta in metadata.items()\n               if verified.get(name, {}).get(\'size\') == meta[\'size\']\n               and cache_source(drive, name, verified[name]).is_file()\n               and cache_source(drive, name, verified[name]).stat().st_size == meta[\'size\']\n               and (not (local/name).is_file() or (local/name).stat().st_size != meta[\'size\'])]\n    if restore:\n        print(f\'Restoring {len(restore)} cached files from Drive to local SSD\', flush=True)\n        def restore_one(name):\n            meta = metadata[name]\n            try:\n                copy_verified(cache_source(drive, name, verified[name]), local/name, meta[\'size\'], meta[\'sha256\'] or verified[name][\'sha256\'])\n                print(\'Restored \' + name, flush=True)\n            except ValueError:\n                verified.pop(name, None)\n                print(\'Drive checksum mismatch; will download a fresh official copy: \' + name, flush=True)\n        with ThreadPoolExecutor(max_workers=2) as pool:\n            list(pool.map(restore_one, restore))\n    download = [name for name, meta in metadata.items()\n                if not (local/name).is_file() or (local/name).stat().st_size != meta[\'size\']]\n    if download:\n        snapshot_download(REPO, revision=REVISION, allow_patterns=download,\n                          local_dir=str(local), max_workers=4)\n    # Keep the model identity for SGLang pipeline discovery without downloading duplicate root weights.\n    snapshot_download(REPO, revision=REVISION, allow_patterns=[\'model_index.json\', \'LICENSE\'],\n                      local_dir=str(local), max_workers=2)\n    print(\'MODEL_READY_ON_LOCAL_SSD: \' + str(local), flush=True)\n    result = {}\n    for name, meta in metadata.items():\n        prior_entry = verified.get(name, {})\n        prior_source = cache_source(drive, name, prior_entry)\n        if prior_source.is_file() and prior_source.stat().st_size == meta[\'size\'] and prior_entry.get(\'sha256\'):\n            result[name] = verified[name]\n            continue\n        wait_for_local_space(meta[\'size\'])\n        try:\n            sha = copy_verified(local/name, drive/name, meta[\'size\'], meta[\'sha256\'])\n        except ValueError:\n            hf_hub_download(REPO, name, revision=REVISION, local_dir=str(local), force_download=True)\n            sha = copy_verified(local/name, drive/name, meta[\'size\'], meta[\'sha256\'])\n        result[name] = {\'size\': meta[\'size\'], \'sha256\': sha}\n        record = {\'repo\': REPO, \'revision\': REVISION, \'files\': {**verified, **result}, \'complete\': False, \'completed_partitions\': completed}\n        temp = drive_manifest.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(record, indent=2)); temp.replace(drive_manifest)\n        print(\'Verified Drive backup \' + name, flush=True)\n    for name in [\'model_index.json\', \'LICENSE\']:\n        source = local/name\n        copy_verified(source, drive/name, source.stat().st_size)\n    record = {\'repo\': REPO, \'revision\': REVISION, \'bytes\': total, \'files\': {**verified, **result}, \'complete\': True, \'partition\': partition, \'completed_partitions\': sorted(set(completed + [partition]))}\n    for folder in [local, drive]:\n        manifest = folder/\'verified-manifest.json\'\n        temp = manifest.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(record, indent=2)); temp.replace(manifest)\n    print(\'DRIVE_BACKUP_COMPLETE: \' + str(drive), flush=True)\n\n\ndef restore_only(local, drive, partition):\n    """Restore an already verified partition without network access or downloads."""\n    record = json.loads((drive/\'verified-manifest.json\').read_text())\n    if record.get(\'repo\') != REPO or record.get(\'revision\') != REVISION:\n        raise RuntimeError(\'Drive cache version does not match this deployment\')\n    completed = record.get(\'completed_partitions\', [record.get(\'partition\', \'FL2VA\')] if record.get(\'complete\') else [])\n    if partition not in completed:\n        raise RuntimeError(partition + \' backup is incomplete; finish its initial download and verification first\')\n    files = {name: meta for name, meta in record.get(\'files\', {}).items()\n             if name.startswith(partition + \'/\')}\n    if not files or partition + \'/model_index.json\' not in files:\n        raise RuntimeError(partition + \' is not cached. Enable download once to prepare it.\')\n    local.mkdir(parents=True, exist_ok=True)\n    need = sum(meta[\'size\'] for name, meta in files.items() if not (local/name).is_file())\n    if shutil.disk_usage(local).free < need + 8 * 1024**3:\n        raise RuntimeError(\'Not enough local SSD space for selected partition\')\n    def restore_one(item):\n        name, meta = item\n        relative = Path(name)\n        if relative.is_absolute() or \'..\' in relative.parts or not meta.get(\'sha256\'):\n            raise RuntimeError(\'Invalid Drive cache manifest entry\')\n        source, target = cache_source(drive, name, meta), local/name\n        if target.is_file() and target.stat().st_size == meta[\'size\']:\n            with target.open(\'rb\') as existing:\n                digest = hashlib.file_digest(existing, \'sha256\').hexdigest()\n            if digest == meta[\'sha256\']:\n                return\n        copy_verified(source, target, meta[\'size\'], meta[\'sha256\'])\n        print(\'Restored \' + name, flush=True)\n    # Bound concurrency to two Drive reads; verify each file before publishing it.\n    pool = ThreadPoolExecutor(max_workers=2)\n    futures = [pool.submit(restore_one, item) for item in files.items()]\n    try:\n        for future in as_completed(futures):\n            future.result()\n    finally:\n        # If any file fails verification, do not keep copying queued files.\n        pool.shutdown(wait=True, cancel_futures=True)\n    for name in (\'model_index.json\', \'LICENSE\'):\n        source = drive/name\n        copy_verified(source, local/name, source.stat().st_size)\n    print(\'MODEL_READY_ON_LOCAL_SSD: \' + str(local), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--local\', type=Path, default=Path(\'/content/MiniMax-H3\'))\n    parser.add_argument(\'--drive\', type=Path, default=Path(\'/content/drive/MyDrive/TapNowLocal/models/MiniMax-H3\')/REVISION)\n    parser.add_argument(\'--partition\', choices=[\'FL2VA\', \'Ref2VA\'], default=\'FL2VA\')\n    parser.add_argument(\'--restore-only\', action=\'store_true\')\n    parser.add_argument(\'--self-test\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.self_test:\n        with tempfile.TemporaryDirectory() as folder:\n            src, dst = Path(folder)/\'src\', Path(folder)/\'dst\'\n            src.write_bytes(b\'weights\')\n            digest = hashlib.sha256(b\'weights\').hexdigest()\n            assert copy_verified(src, dst, 7, digest) == digest\n            assert dst.read_bytes() == b\'weights\'\n            try:\n                copy_verified(src, dst, 8, digest)\n                raise AssertionError(\'Expected integrity error\')\n            except ValueError:\n                assert dst.read_bytes() == b\'weights\'\n                assert not dst.with_name(\'dst.part\').exists()\n        print(\'model cache integrity checks passed\')\n    else:\n        os.environ[\'HF_XET_CHUNK_CACHE_SIZE_BYTES\'] = \'0\'\n        if args.restore_only:\n            restore_only(args.local, args.drive, args.partition)\n        else:\n            cache(args.local, args.drive, args.partition)\n', 'install.sh': '#!/usr/bin/env bash\nset -euo pipefail\npython -m pip install -q uv\nif [ ! -x /content/h3-venv/bin/python ]; then\n  uv venv --python 3.12 /content/h3-venv\nfi\nexport UV_NO_CACHE=1\n# The official source supports disabling optional SRT Rust extensions.\nexport SGLANG_BUILD_RUST_EXTS=none\nuv pip install --python /content/h3-venv/bin/python --prerelease=allow \\\n  \'sglang[diffusion] @ git+https://github.com/sgl-project/sglang.git@45252c64ca8577fb842f953d57f85509a0aa104e#subdirectory=python\'\n# H3 denoising JIT-compiles fused kernels at request time and hard-fails without ninja.\nuv pip install --python /content/h3-venv/bin/python ninja\nif ! command -v ffmpeg >/dev/null || ! command -v ffprobe >/dev/null; then\n  apt-get update -qq\n  apt-get install -y -qq ffmpeg\nfi\n/content/h3-venv/bin/sglang serve --model-type diffusion --help > /content/h3-serve-help.txt\nuv pip freeze --python /content/h3-venv/bin/python > /content/h3-environment.txt\n/content/h3-venv/bin/python "$(dirname "$0")/validate_install.py"\n', 'notebook_session.py': '"""Notebook-owned lifecycle. Import this in the Colab kernel, not the model venv."""\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport threading\nimport time\nimport urllib.request\n\nROOT = Path(\'/content/tapnow-h3\')\nSTATE = Path(\'/content/tapnow-h3-state\')\nDRIVE = Path(\'/content/drive/MyDrive/TapNowLocal\')\nREVISION = \'42ed227ee7df40d41602854ae760620d6eb651fe\'\n_processes = []\n_watcher = None\n\n\ndef prepare(variant=\'fl2va\', allow_download=False, install_env=False):\n    if variant not in (\'fl2va\', \'ref2va\'):\n        raise ValueError(\'Select fl2va or ref2va\')\n    if not DRIVE.parent.is_dir():\n        raise RuntimeError(\'Mount the mert Google Drive first\')\n    if (STATE/\'session.json\').exists():\n        raise RuntimeError(\'A session already exists. Finish it before restarting or changing mode.\')\n    memory = int(subprocess.check_output([\'nvidia-smi\',\'--query-gpu=memory.total\',\'--format=csv,noheader,nounits\'], text=True).splitlines()[0])\n    ram = next(int(line.split()[1]) for line in Path(\'/proc/meminfo\').read_text().splitlines() if line.startswith(\'MemTotal:\'))\n    if memory < 75000 or ram < 120*1024**2:\n        raise RuntimeError(\'This original-weight profile needs an 80 GB-class GPU and at least 120 GiB RAM. Stop here; use a separately tested low-memory profile on smaller machines.\')\n    python = Path(\'/content/h3-venv/bin/python\')\n    if install_env:\n        subprocess.run([\'bash\', str(ROOT/\'install.sh\')], check=True)\n    elif not python.exists():\n        archive = DRIVE/\'session/h3-venv.tar.gz\'\n        if not archive.is_file() or not (DRIVE/\'session/BACKUP_COMPLETE\').is_file():\n            raise RuntimeError(\'No complete saved environment. Set INSTALL_ENV=True for first-time installation.\')\n        subprocess.run([\'tar\',\'-C\',\'/\',\'-xzf\',str(archive)], check=True)\n    # A saved install marker is not evidence that a restored environment still works.\n    Path(\'/content/h3-install.ok\').unlink(missing_ok=True)\n    subprocess.run([str(python), str(ROOT/\'validate_install.py\')], check=True)\n    partition = \'Ref2VA\' if variant==\'ref2va\' else \'FL2VA\'\n    command = [str(python),str(ROOT/\'cache_model.py\'),\'--partition\',partition]\n    if not allow_download:\n        command.append(\'--restore-only\')\n    subprocess.run(command, check=True)\n    # Keep deployment helpers current without copying tokens, notebooks\' outputs, or private config.\n    deployment = DRIVE/\'deployment\'\n    deployment.mkdir(parents=True, exist_ok=True)\n    for file in ROOT.iterdir():\n        if file.suffix in (\'.py\',\'.sh\'):\n            shutil.copy2(file, deployment/file.name)\n    print(\'Ready to start\', partition, \'from local SSD.\')\n\n\ndef release_runtime():\n    from google.colab import drive, runtime\n    drive.flush_and_unmount()\n    runtime.unassign()\n\n\ndef start(variant, token, auto_release=True, idle_seconds=60, open_tunnel=True):\n    global _watcher\n    if _processes or (STATE/\'session.json\').exists():\n        raise RuntimeError(\'Session already started; do not run the start cell twice\')\n    if len(token)<32:\n        raise ValueError(\'Use a temporary token with at least 32 characters\')\n    env = {**os.environ,\'H3_VARIANT\':variant,\'H3_PROXY_TOKEN\':token,\n           \'H3_AUTO_RELEASE\':\'yes\' if auto_release else \'no\',\'H3_IDLE_SECONDS\':str(idle_seconds)}\n    def launch(command, name):\n        with open(\'/content/h3-\'+name+\'.log\',\'w\') as log:\n            process = subprocess.Popen(command,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n        _processes.append(process)\n        return process\n    def wait_ready(url, process, headers=None, timeout=1200):\n        deadline = time.monotonic()+timeout\n        while time.monotonic()<deadline:\n            if process.poll() is not None:\n                raise RuntimeError(\'Service exited; inspect /content/h3-*.log\')\n            try:\n                with urllib.request.urlopen(urllib.request.Request(url,headers=headers or {}), timeout=3) as response:\n                    if response.status==200:\n                        return\n            except Exception:\n                pass\n            time.sleep(3)\n        raise TimeoutError(\'Service did not become ready; inspect logs before retrying\')\n    try:\n        model=launch([\'bash\',str(ROOT/\'start_model.sh\')],\'model\')\n        wait_ready(\'http://127.0.0.1:30020/health\',model)\n        proxy=launch([\'/content/h3-venv/bin/python\',str(ROOT/\'session_service.py\')],\'proxy\')\n        wait_ready(\'http://127.0.0.1:30011/v1/session\',proxy,{\'Authorization\':\'Bearer \'+token},timeout=60)\n        if open_tunnel:\n            env[\'H3_ALLOW_PUBLIC_TUNNEL\']=\'yes\'\n            launch([\'/content/h3-venv/bin/python\',str(ROOT/\'open_tunnel.py\')],\'tunnel\')\n    except BaseException:\n        for process in reversed(_processes):\n            if process.poll() is None:\n                os.killpg(process.pid, signal.SIGTERM)\n        _processes.clear()\n        raise\n    def monitor():\n        while True:\n            time.sleep(3)\n            try:\n                state=json.loads((STATE/\'session.json\').read_text())\n                if state[\'phase\']!=\'release_requested\':\n                    continue\n                # Gateway has stopped admitting work and verified each completed output\'s receipt.\n                release_runtime()\n                return\n            except Exception as error:\n                (STATE/\'release-error.json\').write_text(json.dumps({\'error\':str(error)}))\n                print(\'GPU release did not complete. Check Colab and disconnect/delete runtime manually:\',error)\n                return\n    _watcher=threading.Thread(target=monitor,daemon=True)\n    _watcher.start()\n    print(\'Model and authenticated gateway ready. GPU release watcher is active.\')\n\n\ndef save_environment():\n    """Optional one-time environment backup, without copying the whole /content directory."""\n    session=DRIVE/\'session\'\n    session.mkdir(parents=True, exist_ok=True)\n    python=Path(\'/content/h3-venv/bin/python\').resolve()\n    paths=[\'content/h3-venv\']\n    if not str(python).startswith(\'/usr/\'):\n        paths.append(str(python.parent.parent).lstrip(\'/\'))\n    archive=session/\'h3-venv.tar.gz.part\'\n    subprocess.run([\'tar\',\'-C\',\'/\',\'-czf\',str(archive),*paths],check=True)\n    archive.replace(session/\'h3-venv.tar.gz\')\n    (session/\'BACKUP_COMPLETE\').write_text(time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()))\n    print(\'Saved Python environment for future sessions.\')\n', 'open_tunnel.py': '"""Run only after explicit approval to expose the authenticated video proxy."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport urllib.request\n\n\ndef main():\n    if os.environ.get(\'H3_ALLOW_PUBLIC_TUNNEL\') != \'yes\':\n        raise SystemExit(\'Public exposure is disabled. Confirm the temporary authenticated tunnel first.\')\n    request = urllib.request.Request(\'http://127.0.0.1:30011/health\', headers={\n        \'Authorization\': \'Bearer \' + os.environ[\'H3_PROXY_TOKEN\']})\n    with urllib.request.urlopen(request, timeout=10) as response:\n        if response.status != 200:\n            raise RuntimeError(\'Authenticated proxy is not ready\')\n    with urllib.request.urlopen(\'https://api.github.com/repos/cloudflare/cloudflared/releases/latest\') as response:\n        release = json.load(response)\n    asset = next(a for a in release[\'assets\'] if a[\'name\'] == \'cloudflared-linux-amd64\')\n    digest = asset.get(\'digest\', \'\')\n    if not digest.startswith(\'sha256:\'):\n        raise RuntimeError(\'Official release does not provide a SHA256 digest; verify the binary manually\')\n    binary = Path(\'/content/cloudflared\')\n    temp = binary.with_suffix(\'.part\')\n    hasher = hashlib.sha256()\n    try:\n        with urllib.request.urlopen(asset[\'browser_download_url\']) as src, temp.open(\'wb\') as dst:\n            while chunk := src.read(1024 * 1024):\n                hasher.update(chunk); dst.write(chunk)\n        if hasher.hexdigest() != digest[7:]:\n            raise RuntimeError(\'Cloudflare release checksum mismatch\')\n        temp.chmod(0o700); temp.replace(binary)\n    finally:\n        temp.unlink(missing_ok=True)\n    Path(\'/content/h3-cloudflared-version.json\').write_text(json.dumps({\n        \'version\': release[\'tag_name\'], \'sha256\': digest[7:]}))\n    subprocess.run([str(binary), \'tunnel\', \'--url\', \'http://127.0.0.1:30011\', \'--no-autoupdate\'], check=True)\n\n\nif __name__ == \'__main__\':\n    import sys\n    if \'--self-test\' in sys.argv:\n        from unittest.mock import patch\n        with patch.dict(os.environ, {\'H3_ALLOW_PUBLIC_TUNNEL\': \'no\'}), patch(\'urllib.request.urlopen\', side_effect=AssertionError(\'Must not use network\')):\n            try:\n                main()\n                raise AssertionError(\'Tunnel must stay closed without approval\')\n            except SystemExit as error:\n                assert \'disabled\' in str(error)\n        print(\'tunnel fail-closed check passed (no network exposure)\')\n    else:\n        main()\n', 'session_backup.sh': '#!/usr/bin/env bash\n# Save the environment without copying unrelated /content files or credentials.\nset -euo pipefail\nH3_HELPERS=$(cd -- "$(dirname -- "$0")" && pwd)\nexport PYTHONPATH="$H3_HELPERS${PYTHONPATH:+:$PYTHONPATH}"\n/content/h3-venv/bin/python -c \'import notebook_session; notebook_session.save_environment()\'\necho \'Environment backup complete. Flush Drive before releasing the runtime.\'\n', 'session_restore.sh': '#!/usr/bin/env bash\n# Legacy command-line restore entry; the daily notebook uses notebook_session.prepare.\nset -euo pipefail\nSESSION=/content/drive/MyDrive/TapNowLocal/session\ntest -f "$SESSION/BACKUP_COMPLETE" || { echo \'No complete environment backup\'; exit 1; }\nif [ ! -x /content/h3-venv/bin/python ]; then\n  tar -C / -xzf "$SESSION/h3-venv.tar.gz"\nfi\nH3_HELPERS=$(cd -- "$(dirname -- "$0")" && pwd)\n/content/h3-venv/bin/python "$H3_HELPERS/validate_install.py"\n/content/h3-venv/bin/python "$H3_HELPERS/cache_model.py" --partition "${H3_PARTITION:-FL2VA}" --restore-only\necho \'RESTORE_COMPLETE. Start the model and session gateway from the daily notebook.\'\n', 'session_service.py': '"""Authenticated H3 uploads and download receipts; release is executed by the notebook."""\nimport asyncio\nfrom contextlib import asynccontextmanager, suppress\nimport hashlib\nimport hmac\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport time\nimport uuid\n\nASSET = re.compile(r\'[A-Za-z0-9_-]{1,100}\\.(png|jpg|jpeg|webp|gif|mp4|webm|mov|mp3|wav|ogg|m4a)\')\nJOB = re.compile(r\'[A-Za-z0-9_-]{1,100}\')\n\n\nclass Session:\n    def __init__(self, root, variant, auto_release=False, idle_seconds=60):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.assets = self.root/\'assets\'\n        self.assets.mkdir(exist_ok=True)\n        self.file = self.root/\'session.json\'\n        self.active = 0\n        self.idle_seconds = max(30, idle_seconds)\n        if self.file.exists():\n            self.data = json.loads(self.file.read_text())\n            if self.data[\'variant\'] != variant:\n                raise RuntimeError(\'Finish and release the previous session before changing model mode\')\n        else:\n            self.data = dict(id=uuid.uuid4().hex, variant=variant, autoRelease=auto_release,\n                             phase=\'ready\', jobs={}, lastActivity=time.time())\n            self.save()\n\n    def save(self):\n        temp = self.file.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(self.data))\n        temp.replace(self.file)\n\n    def touch(self):\n        self.data[\'lastActivity\'] = time.time()\n        self.save()\n\n    def blocked(self):\n        return self.active or any(not j.get(\'acknowledged\') and j[\'status\'] != \'failed\'\n                                  for j in self.data[\'jobs\'].values())\n\n    def release(self, automatic=False):\n        if self.data[\'phase\'] != \'ready\' or self.blocked():\n            return False\n        if automatic and (not self.data[\'autoRelease\'] or not self.data[\'jobs\'] or\n                          time.time()-self.data[\'lastActivity\'] < self.idle_seconds):\n            return False\n        self.data[\'phase\'] = \'release_requested\'\n        self.save()\n        return True\n\n    def status(self):\n        jobs = self.data[\'jobs\'].values()\n        return dict(sessionId=self.data[\'id\'], variant=self.data[\'variant\'], phase=self.data[\'phase\'],\n                    tasks=[\'ref2va\'] if self.data[\'variant\']==\'ref2va\' else [\'t2va\', \'fl2va\'],\n                    autoRelease=self.data[\'autoRelease\'], idleSeconds=self.idle_seconds,\n                    pendingResults=sum(not j.get(\'acknowledged\') and j[\'status\']!=\'failed\' for j in jobs),\n                    canRelease=not bool(self.blocked()) and self.data[\'phase\']==\'ready\',\n                    releaseError=json.loads((self.root/\'release-error.json\').read_text()).get(\'error\') if (self.root/\'release-error.json\').exists() else None)\n\n\ndef create_app(root=\'/content/tapnow-h3-state\', variant=None, token=None, transport=None):\n    import httpx\n    from fastapi import FastAPI, Request\n    from fastapi.responses import JSONResponse, StreamingResponse\n    token = token or os.environ.get(\'H3_PROXY_TOKEN\', \'\')\n    if len(token) < 32:\n        raise RuntimeError(\'H3_PROXY_TOKEN must contain at least 32 characters\')\n    variant = variant or os.environ.get(\'H3_VARIANT\', \'fl2va\')\n    if variant not in (\'fl2va\', \'ref2va\'):\n        raise RuntimeError(\'Unsupported model variant\')\n    state = Session(root, variant, os.environ.get(\'H3_AUTO_RELEASE\')==\'yes\',\n                    int(os.environ.get(\'H3_IDLE_SECONDS\', \'60\')))\n\n    @asynccontextmanager\n    async def lifespan(app):\n        async with httpx.AsyncClient(base_url=\'http://127.0.0.1:30020\', timeout=60,\n                                    trust_env=False, transport=transport) as client:\n            app.state.upstream = client\n            async def idle_watch():\n                while True:\n                    await asyncio.sleep(5)\n                    state.release(automatic=True)\n            watcher = asyncio.create_task(idle_watch())\n            try:\n                yield\n            finally:\n                watcher.cancel()\n                with suppress(asyncio.CancelledError):\n                    await watcher\n\n    app = FastAPI(lifespan=lifespan, docs_url=None, redoc_url=None, openapi_url=None)\n    app.state.session = state\n\n    @app.middleware(\'http\')\n    async def authenticate(request, call_next):\n        if not hmac.compare_digest(request.headers.get(\'authorization\', \'\').encode(), (\'Bearer \'+token).encode()):\n            return JSONResponse({\'error\': \'Unauthorized\'}, 401)\n        return await call_next(request)\n\n    @app.get(\'/v1/session\')\n    async def session_status():\n        return state.status()\n\n    @app.post(\'/v1/session/settings\')\n    async def settings(request: Request):\n        body = await request.json()\n        if state.data[\'phase\'] != \'ready\':\n            return JSONResponse({\'error\': \'Session is already releasing\'}, 409)\n        if not isinstance(body.get(\'autoRelease\'), bool):\n            return JSONResponse({\'error\': \'autoRelease must be true or false\'}, 400)\n        state.data[\'autoRelease\'] = body[\'autoRelease\']\n        state.touch()\n        return state.status()\n\n    @app.post(\'/v1/session/release\')\n    async def release():\n        if not state.release():\n            return JSONResponse({\'error\': \'Still generating, transferring, or awaiting saved-result receipts\'}, 409)\n        return {\'phase\': \'release_requested\'}\n\n    @app.post(\'/v1/assets/{name}\')\n    async def upload(name: str, request: Request):\n        if state.data[\'phase\'] != \'ready\':\n            return JSONResponse({\'error\': \'Colab is releasing its GPU; start a new session\'}, 409)\n        if not ASSET.fullmatch(name):\n            return JSONResponse({\'error\': \'Invalid asset name\'}, 400)\n        state.active += 1\n        state.touch()\n        # New generated names prevent concurrent uploads from replacing a referenced asset.\n        target = state.assets/(uuid.uuid4().hex+Path(name).suffix)\n        temp = target.with_suffix(target.suffix+\'.part\')\n        size = 0\n        try:\n            if sum(p.stat().st_size for p in state.assets.iterdir()) > 2*1024**3:\n                return JSONResponse({\'error\': \'Session input storage is full\'}, 413)\n            with temp.open(\'wb\') as output:\n                async for chunk in request.stream():\n                    size += len(chunk)\n                    if size > 100*1024**2:\n                        return JSONResponse({\'error\': \'Reference exceeds 100 MiB\'}, 413)\n                    output.write(chunk)\n            if not size:\n                return JSONResponse({\'error\': \'Empty reference\'}, 400)\n            temp.replace(target)\n            return {\'assetId\': target.name, \'bytes\': size}\n        finally:\n            temp.unlink(missing_ok=True)\n            state.active -= 1\n            state.touch()\n\n    @app.post(\'/v1/videos/{job_id}/receipt\')\n    async def receipt(job_id: str, request: Request):\n        body = await request.json()\n        job = state.data[\'jobs\'].get(job_id)\n        if not job or job[\'status\'] != \'completed\' or not job.get(\'download\'):\n            return JSONResponse({\'error\': \'No complete downloaded result\'}, 409)\n        if body.get(\'sessionId\') != state.data[\'id\'] or body.get(\'bytes\') != job[\'download\'][\'bytes\'] or body.get(\'sha256\') != job[\'download\'][\'sha256\']:\n            return JSONResponse({\'error\': \'Saved-file checksum does not match downloaded result\'}, 409)\n        job[\'acknowledged\'] = True\n        state.touch()\n        return {\'saved\': True}\n\n    @app.api_route(\'/{route:path}\', methods=[\'GET\', \'POST\'])\n    async def proxy(route: str, request: Request):\n        path = \'/\'+route\n        match = re.fullmatch(r\'/v1/videos/([A-Za-z0-9_-]{1,100})(/content)?\', path)\n        submit = request.method==\'POST\' and path==\'/v1/videos\'\n        if not (submit or request.method==\'GET\' and (path in (\'/health\', \'/v1/models\') or match)):\n            return JSONResponse({\'error\': \'Route not exposed\'}, 404)\n        if match and match[1] not in state.data[\'jobs\']:\n            return JSONResponse({\'error\': \'Job does not belong to this session\'}, 404)\n        if submit and state.data[\'phase\'] != \'ready\':\n            return JSONResponse({\'error\': \'Colab is releasing its GPU\'}, 409)\n        state.active += 1\n        state.touch() if submit else None\n        body = bytearray()\n        provisional = None\n        streaming = False\n        try:\n            async for chunk in request.stream():\n                body.extend(chunk)\n                if len(body) > 1024**2:\n                    return JSONResponse({\'error\': \'Request too large\'}, 413)\n            if submit:\n                value = json.loads(body)\n                if value.get(\'task\') not in state.status()[\'tasks\']:\n                    return JSONResponse({\'error\': \'Model mode mismatch; select the matching mode in Colab\'}, 409)\n                conditions = value.get(\'conditions\', [])\n                if not isinstance(conditions, list) or len(conditions)>12:\n                    return JSONResponse({\'error\': \'Invalid conditions\'}, 400)\n                for condition in conditions:\n                    asset = condition.pop(\'assetId\', \'\')\n                    if not ASSET.fullmatch(asset) or not (state.assets/asset).is_file() or \'uri\' in condition:\n                        return JSONResponse({\'error\': \'Upload reference assets before submitting\'}, 400)\n                    condition[\'uri\'] = (state.assets/asset).resolve().as_uri()\n                body = json.dumps(value).encode()\n                provisional = \'pending_\'+uuid.uuid4().hex\n                state.data[\'jobs\'][provisional] = {\'status\': \'submitting\'}\n                state.save()\n            upstream = await app.state.upstream.send(app.state.upstream.build_request(\n                request.method, path, content=bytes(body), headers={\'Content-Type\':\'application/json\', \'Accept-Encoding\':\'identity\'}), stream=True)\n            if match and match[2] and upstream.is_success:\n                streaming = True\n                async def stream():\n                    digest = hashlib.sha256()\n                    size = 0\n                    try:\n                        async for chunk in upstream.aiter_raw():\n                            digest.update(chunk)\n                            size += len(chunk)\n                            yield chunk\n                        length = upstream.headers.get(\'content-length\')\n                        if size >= 12 and (length is None or size == int(length)):\n                            state.data[\'jobs\'][match[1]][\'download\'] = {\'bytes\':size, \'sha256\':digest.hexdigest()}\n                            state.save()\n                    finally:\n                        await upstream.aclose()\n                        state.active -= 1\n                        state.touch()\n                return StreamingResponse(stream(), status_code=upstream.status_code,\n                                         headers={k:v for k,v in upstream.headers.items() if k in (\'content-type\',\'content-length\')})\n            raw = await upstream.aread()\n            await upstream.aclose()\n            try:\n                result = json.loads(raw)\n            except ValueError:\n                return JSONResponse({\'error\':\'Invalid upstream response\'}, 502)\n            if submit:\n                if upstream.is_success and JOB.fullmatch(str(result.get(\'id\',\'\'))):\n                    state.data[\'jobs\'][result[\'id\']] = {\'status\':result.get(\'status\',\'queued\')}\n                    del state.data[\'jobs\'][provisional]\n                elif 400 <= upstream.status_code < 500:\n                    state.data[\'jobs\'][provisional][\'status\'] = \'failed\'\n                state.save()  # Ambiguous submission stays blocked; never release that GPU automatically.\n            elif match and upstream.is_success and result.get(\'status\') in (\'queued\',\'in_progress\',\'completed\',\'failed\'):\n                state.data[\'jobs\'][match[1]][\'status\'] = result[\'status\']\n                state.save()\n            return JSONResponse(result, upstream.status_code)\n        except (ValueError, TypeError, AttributeError):\n            return JSONResponse({\'error\':\'Invalid generation request or response\'}, 400)\n        except httpx.HTTPError:\n            return JSONResponse({\'error\':\'Model service unavailable; submission may be unresolved\'}, 503)\n        finally:\n            if not streaming:\n                state.active -= 1\n                if submit:\n                    state.touch()\n    return app\n\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(create_app(), host=\'127.0.0.1\', port=30011, access_log=False)\n', 'start_model.sh': '#!/usr/bin/env bash\nset -euo pipefail\n# ninja (JIT kernels) lives in the venv; the server is launched without activating it.\nexport PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True\nexport PATH=/content/h3-venv/bin:/usr/local/cuda/bin:$PATH\ncommand -v ninja >/dev/null || { echo \'ninja missing: uv pip install --python /content/h3-venv/bin/python ninja\'; exit 1; }\ntest -f /content/h3-install.ok || { echo \'Install must finish first\'; exit 1; }\nH3_VARIANT=${H3_VARIANT:-fl2va}\ncase "$H3_VARIANT" in\n  fl2va) H3_PARTITION=FL2VA ;;\n  ref2va) H3_PARTITION=Ref2VA ;;\n  *) echo \'H3_VARIANT must be fl2va or ref2va\'; exit 1 ;;\nesac\ntest -f "/content/MiniMax-H3/$H3_PARTITION/model_index.json" || { echo "Native $H3_PARTITION weights missing"; exit 1; }\ncommand -v ffmpeg >/dev/null\ncommand -v ffprobe >/dev/null\nmkdir -p /content/h3-videos\n# shortcut: unbenchmarked 96 GB single-card placement; tune only after a real baseline.\nexec /content/h3-venv/bin/sglang serve \\\n  --model-type diffusion \\\n  --model-path /content/MiniMax-H3 \\\n  --model-id MiniMax-H3 \\\n  --pipeline-class-name MiniMaxH3Pipeline \\\n  --served-model-name MiniMaxAI/MiniMax-H3 \\\n  --model-variant "$H3_VARIANT" --backend sglang \\\n  --num-gpus 1 --tp-size 1 --ulysses-degree 1 \\\n  --performance-mode manual \\\n  --component-residency dit=resident text_encoder=layerwise-offload vae=resident audio_vae=resident \\\n  --pin-cpu-memory false --enable-torch-compile false \\\n  --attention-backend fa --warmup-mode off \\\n  --output-path /content/h3-videos \\\n  --host 127.0.0.1 --port 30020\n', 'validate_install.py': '"""Fail before weight loading if required native H3 code or CLI flags are absent."""\nimport importlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\n\ndef main():\n    for executable in (\'ffmpeg\', \'ffprobe\', \'nvidia-smi\'):\n        if not shutil.which(executable):\n            raise RuntimeError(\'Missing executable: \' + executable)\n    for module in (\'sglang.multimodal_gen.runtime.pipelines.minimax_h3_pipeline\',\n                   \'sglang.multimodal_gen.runtime.entrypoints.http_server\',\n                   \'fastapi\', \'httpx\', \'uvicorn\'):\n        importlib.import_module(module)\n    import ninja\n    cli = Path(sys.executable).parent/\'sglang\'\n    help_text = subprocess.check_output([str(cli), \'serve\', \'--model-type\', \'diffusion\', \'--help\'], text=True, stderr=subprocess.STDOUT)\n    for flag in (\'--component-residency\', \'--model-variant\', \'--pipeline-class-name\', \'--served-model-name\'):\n        if flag not in help_text:\n            raise RuntimeError(\'Installed CLI is missing \' + flag)\n    hardware = subprocess.check_output([\'nvidia-smi\', \'--query-gpu=name,memory.total,driver_version\',\n                                        \'--format=csv,noheader\'], text=True).strip()\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'PyTorch cannot access the GPU\')\n    record = {\'gpu\': hardware, \'python\': sys.version, \'torch\': torch.__version__,\n              \'torch_cuda\': torch.version.cuda, \'device_capability\': torch.cuda.get_device_capability(),\n              \'sglang_commit\': \'45252c64ca8577fb842f953d57f85509a0aa104e\',\n              \'scope\': \'imports, CLI flags and GPU visibility only; no model inference\'}\n    Path(\'/content/h3-install-check.json\').write_text(json.dumps(record, indent=2))\n    Path(\'/content/h3-install.ok\').touch()\n    print(json.dumps(record, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n'}
h3_dir = pathlib.Path('/content/tapnow-h3')
h3_dir.mkdir(exist_ok=True)
for name, source in h3_files.items():
    (h3_dir/name).write_text(source)
sys.path.insert(0, str(h3_dir))
print('Deployment helpers ready')


In [ ]:
# @title ③ 从 Drive 恢复所选模型和环境
# 将子进程输出写入日志，定期显示进度，避免长时间直接输出阻塞页面。
import os, signal, subprocess, sys, time
import notebook_session as h3
prepare_log = pathlib.Path('/content/h3-prepare.log')
prepare_code = ('import sys; sys.path.insert(0, "/content/tapnow-h3"); '
                'import notebook_session as h3; '
                f'h3.prepare({MODE!r}, allow_download={ALLOW_DOWNLOAD!r}, install_env={INSTALL_ENV!r}); '
                + ('h3.save_environment()' if SAVE_ENV else ''))
with prepare_log.open('w') as output:
    prepare_process = subprocess.Popen([sys.executable, '-u', '-c', prepare_code],
                                       stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
try:
    seen = ''
    while prepare_process.poll() is None:
        progress = prepare_log.read_text(errors='replace')[-2000:]
        if progress != seen:
            print(progress or '正在恢复环境和权重…', flush=True)
            seen = progress
        time.sleep(10)
    print(prepare_log.read_text(errors='replace')[-4000:])
    if prepare_process.returncode:
        raise RuntimeError('恢复失败，请查看上方日志；不要继续启动模型。')
except BaseException:
    if prepare_process.poll() is None:
        os.killpg(prepare_process.pid, signal.SIGTERM)
        prepare_process.wait(timeout=15)
    raise


启动会等待模型健康检查。首次加载可能需要较长时间，加载也消耗算力。
输入一个至少 32 字符的临时 Token，同一 Token 填入 TapNow「模型连接 → Colab」。不要把 Token 写进单元格或共享输出。


In [ ]:
# @title ④ 启动所选模型、认证服务和自动释放监视器
import getpass, pathlib
import notebook_session as h3
_token_file = pathlib.Path('/content/.tapnow-h3-token')
if _token_file.is_file():
    h3_token = _token_file.read_text()
    _token_file.unlink()
else:
    h3_token = getpass.getpass('临时访问 Token（至少 32 字符）：')
h3.start(MODE, h3_token, auto_release=AUTO_RELEASE, idle_seconds=IDLE_SECONDS,
         open_tunnel=OPEN_AUTHENTICATED_TUNNEL)


In [ ]:
# @title ⑤ 获取本次 HTTPS 地址（复制到 TapNow）
import re, time
log = pathlib.Path('/content/h3-tunnel.log')
for _ in range(30):
    text = log.read_text() if log.exists() else ''
    urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if urls:
        print('TapNow Colab 服务地址：', urls[-1])
        break
    if not OPEN_AUTHENTICATED_TUNNEL:
        print('未开启隧道')
        break
    time.sleep(2)
else:
    print('隧道尚未就绪，请检查 /content/h3-tunnel.log')


## 使用与关闭

在 TapNow「模型连接 → Colab」填写上面的地址和 Token，保存并连接。
- FL2VA：选择文字或首尾帧，可上传首帧、尾帧或两者。
- Ref2VA：选择全能参考，上传图片／视频／音频，用 @ 素材指定对应关系。
- 当前会话模式不匹配时，画布会拒绝提交，提示到 Colab 切换。

全部任务完成且本地确认文件大小和校验值后，闲置设定时间自动请求释放；笔记本先完成 Drive 写入并卸载，再释放运行环境。
取消接收、提交状态不明或下载失败不会自动释放，以免丢失结果。结果已经在本地但确认失败时，在 Colab 连接设置点击“重新确认本地结果”。
没有任务时也可点击“结束会话并释放 GPU”。若笔记本报告释放失败，检查文件保存情况，再手动“断开连接并删除运行时”。只停模型进程不会停止 GPU 额度消耗。
下次会话必须重新开 GPU、挂载 mert Drive、恢复并启动；临时域名和 Token 可重新设置。
